# 🕸️ Agent Communication Patterns in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Message Passing** - How agents can communicate by appending messages to a shared, growing conversation list
2. **Shared State (Typed Fields)** - How agents can communicate through structured, typed state fields, filled by structured output instead of parsed text
3. **Generator–Critic Refinement** - How a drafter and a critic revise one piece of work through shared state until it passes
4. **Routing and Loops** - How to use conditional edges to loop a graph back on itself until a stopping condition is met

## Where this fits
`03_five_multi_agent_architectures.ipynb` decided *who runs next*. This notebook is about *what agents pass to
each other* while they run.

## Prerequisites
- `03_five_multi_agent_architectures.ipynb`; LangGraph basics (`StateGraph`, nodes, edges, `START`/`END`)
- Familiarity with `TypedDict` state schemas and `Annotated` reducers (e.g. `add_messages`, `operator.add`)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads `OPENAI_API_KEY`; the next cell creates the one chat model every pattern below reuses.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import json
import operator
import warnings
from typing import Literal

from dotenv import find_dotenv, load_dotenv
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field
from typing_extensions import Annotated, TypedDict

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: every agent in every pattern
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 💬 Part 1: Message Passing Pattern

The simplest way for agents to communicate in LangGraph is to share a single, append-only list of messages. Each agent reads everything written so far and appends its own message, so downstream agents automatically see upstream output.

### Key Concepts:
- **Shared message list**: A state field annotated with `add_messages` that accumulates messages instead of overwriting them
- **Implicit context**: Each agent's prompt includes the full message history, so no explicit hand-off logic is needed

### 1.1 📋 `MessagePassingState`

The state schema holds only the growing `messages` list. The graph's fixed edges decide which agent acts next,
so no "phase" field is needed.

In [ ]:
# ============================================================================
# MESSAGE PASSING: State Schema
# ============================================================================
class MessagePassingState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


print("✅ MessagePassingState fields:", list(MessagePassingState.__annotations__))

### 1.2 🏗️ `create_message_passing_pipeline`

Three agents — a researcher, a fact-checker, and a summarizer — run in sequence. Each one reads the entire `messages` list built up so far and appends its own labeled contribution.

In [ ]:
# ============================================================================
# MESSAGE PASSING: Building the Pipeline
# ============================================================================
def create_message_passing_pipeline():
    """Agents communicate by appending messages that others can read."""

    def researcher(state: MessagePassingState) -> dict:
        """Researches the topic and posts findings as a message."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a researcher. Read the user's question, "
                        "research it, and post your findings. Keep it to 2-3 sentences."
                    )
                ),
                *state["messages"],
            ]
        )
        return {
            "messages": [
                AIMessage(
                    content=f"[RESEARCHER]: {response.content}", name="researcher"
                )
            ],
        }

    def fact_checker(state: MessagePassingState) -> dict:
        """Reads the researcher's message and validates the claims."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a fact-checker. Read the researcher's findings "
                        "in the conversation and validate or challenge them. "
                        "Keep it to 2-3 sentences."
                    )
                ),
                *state["messages"],
            ]
        )
        return {
            "messages": [
                AIMessage(
                    content=f"[FACT-CHECKER]: {response.content}", name="fact_checker"
                )
            ],
        }

    def summarizer(state: MessagePassingState) -> dict:
        """Reads all previous messages and creates a final summary."""
        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a summarizer. Read the researcher's findings and "
                        "the fact-checker's review. Produce a final, accurate summary. "
                        "Keep it to 2-3 sentences."
                    )
                ),
                *state["messages"],
            ]
        )
        return {
            "messages": [
                AIMessage(content=f"[SUMMARY]: {response.content}", name="summarizer")
            ],
        }

    graph = StateGraph(MessagePassingState)

    graph.add_node("researcher", researcher)
    graph.add_node("fact_checker", fact_checker)
    graph.add_node("summarizer", summarizer)

    graph.add_edge(START, "researcher")
    graph.add_edge("researcher", "fact_checker")
    graph.add_edge("fact_checker", "summarizer")
    graph.add_edge("summarizer", END)

    return graph.compile()


print("✅ Message passing pipeline defined: researcher -> fact_checker -> summarizer")

### 1.3 ▶️ `demo_message_passing`

Runs the pipeline on a sample question and prints each agent's labeled message in order, showing how the conversation accumulates.

In [ ]:
# ============================================================================
# MESSAGE PASSING: Demo
# ============================================================================
def demo_message_passing():
    """Demo message passing between agents."""
    agent = create_message_passing_pipeline()

    print("Message Passing Demo:\n")
    result = agent.invoke(
        {
            "messages": [
                HumanMessage(content="What are the main benefits of renewable energy?")
            ],
        }
    )

    for msg in result["messages"]:
        if isinstance(msg, AIMessage):
            print(f"{msg.content}\n")

In [ ]:
# ============================================================================
# MESSAGE PASSING: RUN
# ============================================================================
demo_message_passing()

---
## 🗂️ Part 2: Shared State (Typed Fields) Pattern

Instead of a free-form message log, agents can communicate through **structured, typed state fields**. Each agent writes to its own field(s), and downstream agents read those fields directly — no need to parse natural-language history.

### Key Concepts:
- **Typed fields**: Each piece of information (raw data, analysis, recommendations) has its own named, typed slot in state
- **Reducers**: `Annotated[list[dict], operator.add]` lets multiple writes accumulate instead of overwriting
- **Structured output**: each agent's reply comes back as a validated Pydantic object (`llm.with_structured_output(...)`), so typed fields are filled without parsing JSON or regex-matching free text

### 2.1 📋 `SharedFieldsState`

Defines one field per stage of the pipeline: the query, accumulated `raw_data`, the analyst's `analysis` and `confidence_score`, and the advisor's `recommendations`.

In [ ]:
# ============================================================================
# SHARED STATE: State Schema
# ============================================================================
class SharedFieldsState(TypedDict):
    query: str
    # Each agent writes to its own field — others can read it
    raw_data: Annotated[list[dict], operator.add]
    analysis: str
    recommendations: list[str]
    confidence_score: float


print("✅ SharedFieldsState fields:", list(SharedFieldsState.__annotations__))

### 2.2 🏗️ `create_shared_fields_pipeline`

A data collector, an analyst, and an advisor run in sequence. Each one reads only the specific fields it needs from state, and writes back its own fields — no shared conversational context required. Each agent's model call returns a
Pydantic object (`DataPoints`, `Analysis`, `Recommendations`), so a malformed reply fails validation
instead of silently becoming a wrong value.

In [ ]:
# ============================================================================
# SHARED STATE: Building the Pipeline
# ============================================================================
class DataPoint(BaseModel):
    source: str = Field(description="Where the finding comes from (a kind of source, e.g. 'industry survey')")
    finding: str = Field(description="One factual finding, one sentence")


class DataPoints(BaseModel):
    points: list[DataPoint] = Field(description="Exactly 3 data points")


class Analysis(BaseModel):
    analysis: str = Field(description="A brief analysis of the data, 2-3 sentences")
    confidence: float = Field(ge=0.0, le=1.0, description="Confidence in the analysis, from 0.0 to 1.0")


class Recommendations(BaseModel):
    items: list[str] = Field(description="Exactly 3 actionable recommendations, one sentence each")


def create_shared_fields_pipeline():
    """Agents communicate through typed state fields, not messages."""
    collector_llm = llm.with_structured_output(DataPoints)
    analyst_llm = llm.with_structured_output(Analysis)
    advisor_llm = llm.with_structured_output(Recommendations)

    def data_collector(state: SharedFieldsState) -> dict:
        """Collects data and writes to the raw_data field."""
        result = collector_llm.invoke(
            [
                SystemMessage(content="You are a data collector. Given the query, produce 3 data points."),
                HumanMessage(content=state["query"]),
            ]
        )
        return {"raw_data": [point.model_dump() for point in result.points]}  # plain dicts in state

    def analyst(state: SharedFieldsState) -> dict:
        """Reads raw_data field, writes analysis and confidence."""
        data_summary = json.dumps(state["raw_data"], indent=2)
        result = analyst_llm.invoke(
            [
                SystemMessage(content="You are a data analyst. Analyze the collected data and rate your confidence."),
                HumanMessage(content=f"Query: {state['query']}\n\nData:\n{data_summary}"),
            ]
        )
        return {"analysis": result.analysis, "confidence_score": result.confidence}

    def advisor(state: SharedFieldsState) -> dict:
        """Reads analysis + confidence, writes recommendations."""
        result = advisor_llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a strategic advisor. Based on the analysis and "
                        "confidence score, provide 3 actionable recommendations."
                    )
                ),
                HumanMessage(
                    content=(
                        f"Query: {state['query']}\n"
                        f"Analysis: {state['analysis']}\n"
                        f"Confidence: {state['confidence_score']}"
                    )
                ),
            ]
        )
        return {"recommendations": result.items}

    graph = StateGraph(SharedFieldsState)

    graph.add_node("data_collector", data_collector)
    graph.add_node("analyst", analyst)
    graph.add_node("advisor", advisor)

    graph.add_edge(START, "data_collector")
    graph.add_edge("data_collector", "analyst")
    graph.add_edge("analyst", "advisor")
    graph.add_edge("advisor", END)

    return graph.compile()


print("✅ Shared state pipeline defined: data_collector -> analyst -> advisor")

### 2.3 ▶️ `demo_shared_state`

Runs the pipeline on a sample business question and prints the collected data, the analysis, the confidence score, and the final recommendations.

In [ ]:
# ============================================================================
# SHARED STATE: Demo
# ============================================================================
def demo_shared_state():
    """Demo shared state fields between agents."""
    agent = create_shared_fields_pipeline()

    print("Shared State Demo:\n")
    result = agent.invoke(
        {
            "query": "Should a small business invest in AI automation in 2026?",
            "raw_data": [],
            "analysis": "",
            "recommendations": [],
            "confidence_score": 0.0,
        }
    )

    print(f"Data collected: {len(result['raw_data'])} points")
    for d in result["raw_data"]:
        print(f"  - [{d['source']}] {d['finding'][:80]}...")

    print(f"\nAnalysis: {result['analysis'][:200]}...")
    print(f"Confidence: {result['confidence_score']}")

    print(f"\nRecommendations:")
    for i, rec in enumerate(result["recommendations"], 1):
        print(f"  {i}. {rec}")

    typed = (
        all(set(d) == {"source", "finding"} for d in result["raw_data"])
        and isinstance(result["confidence_score"], float)
        and 0.0 <= result["confidence_score"] <= 1.0
        and all(isinstance(r, str) for r in result["recommendations"])
    )
    print(f"\n{'✅' if typed else '❌'} every field holds the type its schema declares — nothing was parsed from text")

In [ ]:
# ============================================================================
# SHARED STATE: RUN
# ============================================================================
demo_shared_state()

---
## 🔁 Part 3: Generator–Critic Refinement

A **drafter** (the generator) writes a short paragraph and a **critic** reviews it. Both read and write the same
state — the running lists of `drafts` and `critiques` — and a conditional edge sends the work back to the drafter
until the critic approves it or an iteration cap ends the loop.

The critic does not hand down a yes/no verdict. For each of three criteria it **copies the sentence from the draft
that meets it**, or writes `none`. Code then checks that every quote is really in the draft, and that the
feature quote names a real LangGraph feature from a short list, and computes the verdict — so the loop's exit
never rests on the model's say-so.

> **Not a blackboard**: this loop is sometimes called a blackboard, but the order here is fixed (draft → critique
> → draft …); only the loop's exit is decided at run time. A *blackboard* has a controller that picks which of
> several specialists acts next from what is on the shared board — see
> [`01_blackboard_intro.ipynb`](../../m26_collaboration_parallel_agents/notebooks/01a_blackboard_intro.ipynb) and
> [`02_blackboard_research_team.ipynb`](../../m26_collaboration_parallel_agents/optional/01b_blackboard_research_team.ipynb) in module 08. For a larger
> generator–critic loop (a developer agent and a tester agent), see
> [`11_developer_vs_tester_agents.ipynb`](../../m27_multi_agent_builds_deep_agents/optional/04a_developer_vs_tester_agents.ipynb) in module 10.

### Key Insight:
> The `drafts` and `critiques` lists are a shared history both agents can see. The drafter always gets the
> previous draft *and* the feedback on it, so it revises instead of starting over.

### 3.1 📋 `RefinementState`

Alongside the message log, the state holds the `topic`, the running lists of `drafts` and `critiques`, an
`iteration` counter, and two flags: `is_approved` (the critic's computed verdict) and `capped` (the loop hit
`MAX_ITERATIONS` without approval).

In [ ]:
# ============================================================================
# GENERATOR–CRITIC: State Schema
# ============================================================================
MAX_ITERATIONS = 3  # the loop ends here even if the critic never approves


class RefinementState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    # Shared fields both agents read and write
    topic: str
    drafts: Annotated[list[str], operator.add]
    critiques: Annotated[list[str], operator.add]
    iteration: int
    is_approved: bool
    capped: bool


print("✅ RefinementState fields:", list(RefinementState.__annotations__))

### 3.2 🏗️ `create_refinement_loop`

The `drafter` reads the latest draft and critique and writes an improved draft. The `critic` fills a `Critique`
object: one evidence field per criterion plus feedback. `missing_criteria()` decides in code which criteria are
not met — a quote that is `none`, that does not appear in the draft, or (for the feature) that names no feature
from `LANGGRAPH_FEATURES` counts as missing. A conditional edge,
`route_after_critic`, loops back to the drafter until nothing is missing or `MAX_ITERATIONS` is reached.

In [ ]:
# ============================================================================
# GENERATOR–CRITIC: Building the Loop
# ============================================================================
class Critique(BaseModel):
    """Evidence for each criterion, copied from the draft, plus feedback."""
    feature_quote: str = Field(description="The sentence that names a specific LangGraph feature (e.g. checkpointer, "
                                           "conditional edges, Command, subgraphs), copied exactly, or 'none'")
    example_quote: str = Field(description="The sentence that gives a concrete multi-agent use case, copied exactly, or 'none'")
    tradeoff_quote: str = Field(description="The sentence that states a limitation, cost or trade-off of LangGraph, "
                                            "copied exactly, or 'none'")
    feedback: str = Field(description="What the writer must change; name every criterion whose quote is 'none'")


CRITERIA = {"feature_quote": "a specific LangGraph feature (e.g. checkpointers, conditional edges, subgraphs, interrupts)",
            "example_quote": "a concrete use case", "tradeoff_quote": "a limitation or trade-off"}
# Code-checkable part of criterion 1: the quote must name one of these
LANGGRAPH_FEATURES = ("checkpoint", "conditional edge", "command", "subgraph", "interrupt", "stategraph",
                      "reducer", "persistence", "streaming", "human-in-the-loop")


def missing_criteria(critique: Critique, draft: str) -> list[str]:
    """Criteria without real evidence: 'none', a quote that is not in the draft, or no known feature named."""
    norm = lambda s: " ".join(s.lower().split()).strip(" .\"'")  # noqa: E731
    missing = []
    for field, label in CRITERIA.items():
        quote = norm(getattr(critique, field))
        if quote in ("", "none") or quote not in norm(draft):
            missing.append(label)
        elif field == "feature_quote" and not any(f in quote for f in LANGGRAPH_FEATURES):
            missing.append(label)
    return missing


def create_refinement_loop():
    """A drafter writes, a critic reviews with evidence, and they iterate until approved or capped."""
    critic_llm = llm.with_structured_output(Critique)

    def drafter(state: RefinementState) -> dict:
        """Reads the latest draft and critique, writes an improved draft."""
        context_parts = [f"Topic: {state['topic']}"]
        if state["drafts"]:
            context_parts.append(f"Previous draft: {state['drafts'][-1]}")
        if state["critiques"]:
            context_parts.append(f"Feedback to address: {state['critiques'][-1]}")
        context = "\n".join(context_parts)

        response = llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a skilled writer. Write or revise a short paragraph "
                        "(3-4 sentences) based on the topic and any feedback provided. "
                        "If there's feedback, directly address it in your revision. "
                        "Reply with the paragraph only."
                    )
                ),
                HumanMessage(content=context),
            ]
        )

        return {
            "drafts": [response.content],
            "messages": [
                AIMessage(
                    content=f"[DRAFTER iteration {state['iteration'] + 1}]: {response.content}",
                    name="drafter",
                )
            ],
            "iteration": state["iteration"] + 1,
        }

    def critic(state: RefinementState) -> dict:
        """Reads the latest draft, quotes the evidence for each criterion; code computes the verdict."""
        latest_draft = state["drafts"][-1]

        critique = critic_llm.invoke(
            [
                SystemMessage(
                    content=(
                        "You are a strict editor. A paragraph is publishable only if it (1) names a specific LangGraph "
                        "feature such as checkpointers, conditional edges, Command, subgraphs or interrupts, (2) gives a concrete multi-agent use case, and (3) states a limitation, cost or "
                        "trade-off of using LangGraph. For each criterion copy the sentence that meets it exactly as "
                        "written, or write 'none'. Praise does not count as a trade-off."
                    )
                ),
                HumanMessage(content=f"Topic: {state['topic']}\nDraft: {latest_draft}"),
            ]
        )

        missing = missing_criteria(critique, latest_draft)
        approved = not missing
        capped = not approved and state["iteration"] >= MAX_ITERATIONS
        verdict = "APPROVED" if approved else ("CAP REACHED" if capped else "REVISION NEEDED")

        result = {
            "is_approved": approved,
            "capped": capped,
            "messages": [
                AIMessage(
                    content=f"[CRITIC]: {verdict} - missing: {missing or 'nothing'} - {critique.feedback}",
                    name="critic",
                )
            ],
        }
        if not approved:
            result["critiques"] = [f"Missing: {', '.join(missing)}. {critique.feedback}"]
        return result

    def route_after_critic(state: RefinementState) -> Literal["drafter", "end"]:
        """Loop back to the drafter until approved or capped."""
        if state["is_approved"] or state["capped"]:
            return "end"
        return "drafter"

    graph = StateGraph(RefinementState)

    graph.add_node("drafter", drafter)
    graph.add_node("critic", critic)

    graph.add_edge(START, "drafter")
    graph.add_edge("drafter", "critic")
    graph.add_conditional_edges(
        "critic", route_after_critic, {"drafter": "drafter", "end": END}
    )

    return graph.compile()


print(f"✅ Generator–critic loop defined: drafter <-> critic (until approved, at most {MAX_ITERATIONS} drafts)")

### 3.3 ▶️ `demo_refinement`

Runs the drafter/critic loop on a sample topic and prints every message, how many drafts it took, how the loop
ended, and the final draft.

In [ ]:
# ============================================================================
# GENERATOR–CRITIC: Demo
# ============================================================================
def demo_refinement():
    """Demo generator–critic refinement."""
    agent = create_refinement_loop()

    print("Generator–Critic Demo:\n")
    result = agent.invoke(
        {
            "messages": [],
            "topic": "Why LangGraph is great for building multi-agent systems",
            "drafts": [],
            "critiques": [],
            "iteration": 0,
            "is_approved": False,
            "capped": False,
        }
    )

    print("Conversation:")
    for msg in result["messages"]:
        if isinstance(msg, AIMessage):
            print(f"\n{msg.content}")

    print(f"\nFinal draft:\n{result['drafts'][-1]}")

    ended = "approval (every criterion quoted from the draft)" if result["is_approved"] else f"the cap ({MAX_ITERATIONS} drafts)"
    print(f"\nDrafts written: {result['iteration']} · ended by {ended}")
    print(f"{'✅' if result['iteration'] > 1 else '❌'} the critic's feedback triggered at least one revision")
    return result


refinement_result = demo_refinement()

Read the critic's messages: the first draft usually praises LangGraph without a trade-off, so the critic quotes
`none` for it and the drafter revises with that feedback. The ✅ line and the "ended by" line are computed in
code from the state. The cap (`MAX_ITERATIONS`) is what guarantees the loop ends even if the critic never approves.

---
## 📝 Summary

In this notebook, we learned three ways for agents in a LangGraph graph to communicate with each other:

### 1. Message Passing
- Agents share a single `messages` list (`Annotated[list[BaseMessage], add_messages]`)
- Each agent reads the full history and appends its own labeled message
- Simple and transparent, but the LLM must parse conversational context each time

### 2. Shared State (Typed Fields)
- Agents communicate through structured, typed state fields instead of free text
- `with_structured_output` fills those fields with validated Pydantic objects — no JSON or regex parsing
- Reducers like `operator.add` let multiple agents accumulate into the same field
- More structured and easier to validate than raw messages, at the cost of a more rigid schema

### 3. Generator–Critic Refinement
- A drafter and a critic read and write shared state (`drafts`, `critiques`) and iterate
- The critic quotes its evidence per criterion; code checks the quotes and computes the verdict
- A conditional edge (`route_after_critic`) loops the graph until approval, and `MAX_ITERATIONS` guarantees it ends
- It is not a blackboard: the order is fixed, and no controller chooses who acts next

### Next Steps
- **`05_supervisor_workflow_with_human_review.ipynb`**: a supervisor that routes to specialist agents until an article passes its checks, with a human review step
- **Module 08 — [`01_blackboard_intro.ipynb`](../../m26_collaboration_parallel_agents/notebooks/01a_blackboard_intro.ipynb)**: a real blackboard, where a controller picks the next specialist